# 07 · DQN Training: Flat Reward vs LBD Reward

## Colab setup

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/UNIBO/Deep Learning/DL_3C")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, str(PROJECT_DIR))
else:
    PROJECT_DIR = Path.cwd()

IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
IMAGES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Working folder:", Path.cwd())
print("Files:", os.listdir())

## Imports

In [ ]:
import copy
import pickle
import random
import time
from collections import deque

import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from cdcl import Solver, solve
from qnet import QNetwork, state_graph, batch_graphs, to_device, literal_from_action, choose_action, q_literal

sns.set_theme(style="whitegrid", context="notebook")

## Settings

In [ ]:
SEED = 42
DIM = 64
ROUNDS = 4
EPISODES = 1500
MAX_DECISIONS = 300
VAL_MAX_DECISIONS = 1000
GAMMA = 0.99
LR = 1e-4
BATCH_SIZE = 32
BUFFER_SIZE = 20000
WARMUP_STEPS = 500
TARGET_UPDATE = 500
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 10000
EVAL_EVERY = 100
STEP_PENALTY = -0.1
LBD_WEIGHT = 0.02
GLUE_LBD = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data

In [ ]:
with open("rl_data.pkl", "rb") as f:
    rl_data = pickle.load(f)

with open("baselines.pkl", "rb") as f:
    baselines = pickle.load(f)

rl_train = rl_data["train"]
rl_val = rl_data["val"]
val_reference = baselines["val_reference"]

print("Training formulas:  ", len(rl_train))
print("Validation formulas:", len(rl_val))
print("Validation reference:", {k: round(v, 1) for k, v in val_reference.items()})

## Reward functions

In [ ]:
def flat_reward(lbds):
    return STEP_PENALTY


def lbd_reward(lbds):
    return STEP_PENALTY - LBD_WEIGHT * sum(max(lbd - GLUE_LBD, 0) for lbd in lbds)


REWARDS = {"flat": flat_reward, "lbd": lbd_reward}

## Reward examples

In [ ]:
examples = [[], [2], [2, 2], [5], [3, 8], [10, 12, 9]]

print(f"{'learned LBDs':>15s} | {'flat':>6s} | {'lbd':>6s}")
for lbds in examples:
    print(f"{str(lbds):>15s} | {flat_reward(lbds):6.2f} | {lbd_reward(lbds):6.2f}")

## Epsilon schedule

In [ ]:
def epsilon_at(step):
    fraction = min(step / EPS_DECAY_STEPS, 1.0)
    return EPS_START + fraction * (EPS_END - EPS_START)


steps = list(range(0, 2 * EPS_DECAY_STEPS, 100))
plt.figure(figsize=(6, 3))
sns.lineplot(x=steps, y=[epsilon_at(s) for s in steps])
plt.xlabel("Training step")
plt.ylabel("Epsilon")
plt.tight_layout()
plt.show()

## Q-values of chosen actions

In [ ]:
def q_of_actions(model, graphs, actions):
    batch = to_device(batch_graphs(graphs), device)
    q = model(batch).view(-1)

    offsets = []
    total = 0
    for graph in graphs:
        offsets.append(total)
        total += graph["n_vars"]

    index = torch.tensor([2 * o + a for o, a in zip(offsets, actions)], device=device)
    return q[index]

## Best Q-value per state

In [ ]:
def max_q(model, graphs):
    batch = to_device(batch_graphs(graphs), device)
    q = model(batch).view(-1)
    action_graph = batch["var_graph"].repeat_interleave(2)
    best = torch.full((len(graphs),), float("-inf"), device=device)
    return best.scatter_reduce(0, action_graph, q, reduce="amax")

## Training step

In [ ]:
def train_step(model, target, optimizer, buffer):
    batch = random.sample(buffer, BATCH_SIZE)

    states = [t[0] for t in batch]
    actions = [t[1] for t in batch]
    rewards = torch.tensor([t[2] for t in batch], dtype=torch.float, device=device)
    next_graphs = [t[3] for t in batch if t[3] is not None]
    not_done = torch.tensor([t[3] is not None for t in batch], device=device)

    q = q_of_actions(model, states, actions)

    with torch.no_grad():
        next_values = torch.zeros(BATCH_SIZE, device=device)
        if next_graphs:
            next_values[not_done] = max_q(target, next_graphs)
        targets = rewards + GAMMA * next_values

    loss = nn.functional.smooth_l1_loss(q, targets)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()

## Validation

In [ ]:
def validate(model):
    model.eval()
    choose = q_literal(model, device)
    decisions = []
    for p in rl_val:
        solver = solve(p["n_vars"], p["clauses"], choose, VAL_MAX_DECISIONS)
        decisions.append(solver.decisions)
    model.train()
    return sum(decisions) / len(decisions)

## Episode

In [ ]:
def run_episode(problem, model, target, optimizer, buffer, reward_fn, counters):
    solver = Solver(problem["n_vars"], problem["clauses"])
    solver.start()
    if solver.result is not None:
        return None

    graph = state_graph(solver)
    total_reward = 0
    losses = []

    while solver.result is None and solver.decisions < MAX_DECISIONS:
        action = choose_action(model, graph, epsilon_at(counters["step"]), device)
        lbds = solver.step(literal_from_action(graph, action))
        reward = reward_fn(lbds)
        next_graph = None if solver.result is not None else state_graph(solver)

        buffer.append((graph, action, reward, next_graph))
        graph = next_graph
        total_reward += reward
        counters["step"] += 1

        if len(buffer) >= WARMUP_STEPS:
            losses.append(train_step(model, target, optimizer, buffer))

        if counters["step"] % TARGET_UPDATE == 0:
            target.load_state_dict(model.state_dict())

    return {
        "decisions": solver.decisions,
        "return": total_reward,
        "loss": sum(losses) / len(losses) if losses else float("nan"),
        "solved": solver.result is not None,
    }

## Training loop

In [ ]:
def train(mode):
    random.seed(SEED)
    torch.manual_seed(SEED)

    model = QNetwork(DIM, ROUNDS).to(device)
    target = copy.deepcopy(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    buffer = deque(maxlen=BUFFER_SIZE)
    counters = {"step": 0}
    reward_fn = REWARDS[mode]

    history = {"decisions": [], "return": [], "loss": [], "solved": [], "val_episode": [], "val_decisions": []}
    best_val = float("inf")
    start = time.perf_counter()

    for episode in range(1, EPISODES + 1):
        problem = random.choice(rl_train)
        result = run_episode(problem, model, target, optimizer, buffer, reward_fn, counters)

        if result is not None:
            for key in ["decisions", "return", "loss", "solved"]:
                history[key].append(result[key])

        if episode % EVAL_EVERY == 0:
            val = validate(model)
            history["val_episode"].append(episode)
            history["val_decisions"].append(val)
            if val < best_val:
                best_val = val
                torch.save(model.state_dict(), MODELS_DIR / f"qnet_{mode}.pt")
            recent = history["decisions"][-EVAL_EVERY:]
            print(f"{mode:4s} | episode {episode:5d} | steps {counters['step']:6d} | epsilon {epsilon_at(counters['step']):.2f} | train decisions {sum(recent) / len(recent):6.1f} | val decisions {val:6.1f}")

    history["best_val"] = best_val
    history["steps"] = counters["step"]
    history["minutes"] = (time.perf_counter() - start) / 60
    return history

## Train with flat reward

In [ ]:
histories = {}
histories["flat"] = train("flat")

print(f"Best validation decisions: {histories['flat']['best_val']:.1f}")
print(f"Training time: {histories['flat']['minutes']:.1f} minutes")

## Train with LBD reward

In [ ]:
histories["lbd"] = train("lbd")

print(f"Best validation decisions: {histories['lbd']['best_val']:.1f}")
print(f"Training time: {histories['lbd']['minutes']:.1f} minutes")

## Decisions during training

In [ ]:
plt.figure(figsize=(8, 4))
for mode, history in histories.items():
    smooth = pd.Series(history["decisions"]).rolling(50, min_periods=1).mean()
    sns.lineplot(x=smooth.index, y=smooth.values, estimator=None, label=f"{mode} reward")
plt.xlabel("Episode")
plt.ylabel("Decisions per episode (moving average)")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "dqn_train_decisions.png", dpi=150)
plt.show()

## Validation curves

In [ ]:
plt.figure(figsize=(8, 4))
for mode, history in histories.items():
    sns.lineplot(x=history["val_episode"], y=history["val_decisions"], marker="o", label=f"{mode} reward")
plt.axhline(val_reference["VSIDS"], color="gray", linestyle="--", label="VSIDS")
plt.axhline(val_reference["Random"], color="black", linestyle=":", label="Random")
plt.xlabel("Episode")
plt.ylabel("Mean validation decisions")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "dqn_validation.png", dpi=150)
plt.show()

## Loss curves

In [ ]:
plt.figure(figsize=(8, 4))
for mode, history in histories.items():
    smooth = pd.Series(history["loss"]).rolling(50, min_periods=1).mean()
    sns.lineplot(x=smooth.index, y=smooth.values, estimator=None, label=f"{mode} reward")
plt.xlabel("Episode")
plt.ylabel("Mean Huber loss (moving average)")
plt.yscale("log")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "dqn_loss.png", dpi=150)
plt.show()

## Episode returns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (mode, history) in zip(axes, histories.items()):
    smooth = pd.Series(history["return"]).rolling(50, min_periods=1).mean()
    sns.lineplot(x=smooth.index, y=smooth.values, estimator=None, ax=ax)
    ax.set_title(f"{mode} reward")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Return (moving average)")

plt.tight_layout()
plt.savefig(IMAGES_DIR / "dqn_returns.png", dpi=150)
plt.show()

## Training summary

In [ ]:
print(f"{'reward':>6s} | {'episodes':>8s} | {'steps':>7s} | {'solved share':>12s} | {'best val':>8s} | {'minutes':>7s}")
for mode, h in histories.items():
    solved_share = sum(h["solved"]) / len(h["solved"])
    print(f"{mode:>6s} | {len(h['decisions']):8d} | {h['steps']:7d} | {solved_share:12.3f} | {h['best_val']:8.1f} | {h['minutes']:7.1f}")

print(f"VSIDS on validation: {val_reference['VSIDS']:.1f}")

## Save

In [ ]:
with open("dqn_history.pkl", "wb") as f:
    pickle.dump(histories, f)

print("Saved qnet_flat.pt, qnet_lbd.pt, dqn_history.pkl and the training plots")